# WaveBot V3 — Swing Reversal Backtest

**4H 3-candle reversal strategy with ATR trailing stop**

This notebook runs the V3 strategy against OANDA data:
1. **Setup** — Clone repo, install deps, mount Drive
2. **Load Data** — Reuses cached OANDA Parquet files from V1 (H4 + D only)
3. **Run V3 Backtest** — 3-candle reversal detection, trailing SL, double-down/flip
4. **Results** — Equity curve, trade log, performance metrics
5. **V1 vs V3 Comparison** — Side-by-side metrics

> V3 trades less frequently but targets high-conviction reversals.
> No take profit — exits via opposite signal or ATR trailing stop.
> Includes failed-reversal double-down and confirmed-reversal flip logic.

---
## Step 1 — Setup

In [ ]:
import os

REPO_URL = "https://github.com/Unseengap/wavebot-v1.git"
REPO_DIR = "/content/wavebot-v1"

if os.path.exists(REPO_DIR):
    print("Repo already cloned — pulling latest changes...")
    !cd {REPO_DIR} && git pull origin main
else:
    print("Cloning repo...")
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log --oneline -5
print("\n✅ Code ready.")

In [ ]:
!pip install oandapyV20 numpy pandas matplotlib plotly requests pytz pyyaml python-dotenv -q

from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = "/content/drive/MyDrive/wavebot"
DRIVE_DATA = f"{DRIVE_BASE}/data"
DRIVE_RESULTS_V3 = f"{DRIVE_BASE}/results_v3"

os.makedirs(DRIVE_RESULTS_V3, exist_ok=True)

print(f"Data dir:     {DRIVE_DATA}")
print(f"V3 Results:   {DRIVE_RESULTS_V3}")
print("\n✅ Drive mounted.")

---
## Step 2 — Load Data

V3 only needs **H4** and **D** timeframes.
Reuses the same cached Parquet files from V1's OANDA download.
If not cached yet, downloads fresh from OANDA.

In [ ]:
# --- OANDA Credentials (only needed if data not cached) ---
try:
    from google.colab import userdata
    OANDA_TOKEN = userdata.get('OANDA_TOKEN')
    OANDA_ACCOUNT = userdata.get('OANDA_ACCOUNT')
    print("✅ Loaded credentials from Colab Secrets.")
except:
    OANDA_TOKEN = None
    OANDA_ACCOUNT = None
    print("⚠️  No credentials found — will only work if data is cached on Drive.")

INSTRUMENT = "EUR_USD"
DATA_START = "2020-01-01T00:00:00Z"
DATA_END = "2024-12-31T00:00:00Z"

print(f"\nInstrument: {INSTRUMENT}")
print(f"Period:     {DATA_START[:10]} to {DATA_END[:10]}")

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

import pandas as pd
import numpy as np

V3_TIMEFRAMES = ["H4", "D"]
candle_data = {}

for tf in V3_TIMEFRAMES:
    cache_file = f"{DRIVE_DATA}/{INSTRUMENT}_{tf}.parquet"

    if os.path.exists(cache_file):
        df = pd.read_parquet(cache_file)
        print(f"  {tf}: Loaded from cache — {len(df):,} candles")
    else:
        if OANDA_TOKEN is None:
            raise RuntimeError(
                f"No cached data for {tf} and no OANDA credentials. "
                "Run the V1 notebook first to download data, or add OANDA_TOKEN to Colab Secrets."
            )
        print(f"  {tf}: Downloading from OANDA...")
        from src.data.oanda_client import OandaClient
        client = OandaClient(OANDA_TOKEN, OANDA_ACCOUNT, environment="practice")
        df = client.collect_range(INSTRUMENT, tf, start=DATA_START, end=DATA_END)
        df.to_parquet(cache_file, index=False)
        print(f"  {tf}: Downloaded {len(df):,} candles → saved to Drive")

    candle_data[tf] = df

print(f"\n✅ Data ready:")
for tf, df in candle_data.items():
    print(f"   {tf:>4}: {len(df):>8,} candles  |  {str(df['time'].iloc[0])[:10]} → {str(df['time'].iloc[-1])[:10]}")

In [ ]:
def normalize_ohlc(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize OANDA column names to standard OHLC.
    OANDA Parquet files use 'open_mid', 'high_mid', etc.
    V3 expects 'open', 'high', 'low', 'close'.
    """
    out = pd.DataFrame()
    out["time"] = df["time"]

    if "open_mid" in df.columns:
        out["open"] = df["open_mid"].astype(float)
        out["high"] = df["high_mid"].astype(float)
        out["low"] = df["low_mid"].astype(float)
        out["close"] = df["close_mid"].astype(float)
    elif "open" in df.columns:
        out["open"] = df["open"].astype(float)
        out["high"] = df["high"].astype(float)
        out["low"] = df["low"].astype(float)
        out["close"] = df["close"].astype(float)
    else:
        raise ValueError(f"Unknown columns: {list(df.columns)}")

    return out

h4_df = normalize_ohlc(candle_data["H4"])
d_df = normalize_ohlc(candle_data["D"])

print(f"H4: {len(h4_df):,} candles | Columns: {list(h4_df.columns)}")
print(f" D: {len(d_df):,} candles  | Columns: {list(d_df.columns)}")
print()
h4_df.head(3)

---
## Step 3 — Run V3 Backtest

**Strategy:** Detect 3-candle reversal patterns on 4H.
**Entries:** Immediate on Candle 3 close confirmation.
**Exits:** ATR trailing stop or opposite reversal signal.
**No take profit** — ride the trend until stopped out or flipped.

### How it works:
1. **Candle 1** — Strong trend candle (e.g. bearish sell-off)
2. **Candle 2** — Closes in the opposite direction of C1 (first reversal sign)
3. **Candle 3** — Closes at or beyond C2's close (confirmation). Even if C3 dips against the direction intra-bar, closing back = "rejection confirmation"

### Position Management:
- **Failed reversal → Double down**: If opposing candles form but never close beyond your entry reference → add to position on next same-direction signal
- **Confirmed reversal → Flip**: If a candle closes beyond the reference → close position + open opposite

In [ ]:
from src.strategy.v3.backtest_runner import V3BacktestEngine
from src.backtest.metrics import calculate_metrics

# --- V3 Configuration ---
v3_config = {
    "instrument": INSTRUMENT,
    "pip_size": 0.0001,
    "pip_value_per_unit": 0.0001,
    "initial_balance": 10000.0,
    "risk_fraction": 0.01,
    # Pattern detection
    "min_body_ratio": 0.3,          # Reject doji candles as Candle 1
    # ATR trailing stop
    "atr_period": 14,
    "atr_multiplier": 2.0,          # Trail by 2 × ATR(14) on H4
    "sl_buffer_pips": 2.0,          # Buffer beyond pattern extreme
    # Position management
    "max_open_v3_trades": 2,
    "double_down_enabled": True,
    "max_double_downs": 1,
    # Risk limits
    "max_daily_drawdown": 0.02,
    "max_total_drawdown": 0.08,
}

print("V3 Configuration:")
for k, v in v3_config.items():
    print(f"  {k:<25} = {v}")
print()

In [ ]:
import time as _time

engine = V3BacktestEngine(**v3_config)

print(f"Running V3 backtest on {INSTRUMENT}...")
print(f"  H4 candles: {len(h4_df):,}")
print(f"   D candles: {len(d_df):,}")
print()

t0 = _time.time()
v3_trades = engine.run(h4_df, d_df)
elapsed = _time.time() - t0

print(f"\n✅ Backtest complete in {elapsed:.2f}s")
print(f"   Total trades: {len(v3_trades)}")
print(f"   Final balance: ${engine.balance:,.2f}")

In [ ]:
# Calculate trading days from the data range
trading_days = (pd.Timestamp(h4_df['time'].iloc[-1]) -
                pd.Timestamp(h4_df['time'].iloc[0])).days
trading_days = max(1, trading_days)

v3_metrics = calculate_metrics(
    v3_trades,
    initial_balance=v3_config["initial_balance"],
    instrument=INSTRUMENT,
    start=DATA_START[:10],
    end=DATA_END[:10],
    trading_days=trading_days,
)

print("=" * 65)
print("  SWING REVERSAL V3 — PERFORMANCE METRICS")
print("=" * 65)
print(f"  Total Trades:      {v3_metrics.total_trades}")
print(f"  Win Rate:          {v3_metrics.win_rate:.1%}")
print(f"  Profit Factor:     {v3_metrics.profit_factor:.2f}")
print(f"  Total Pips:        {v3_metrics.total_pips:+.1f}")
print(f"  Avg Win:           {v3_metrics.avg_win_pips:+.1f} pips")
print(f"  Avg Loss:          {v3_metrics.avg_loss_pips:.1f} pips")
print(f"  Avg R:R Achieved:  {v3_metrics.avg_rr_achieved:.2f}")
print(f"  Sharpe Ratio:      {v3_metrics.sharpe_ratio:.2f}")
print(f"  Calmar Ratio:      {v3_metrics.calmar_ratio:.2f}")
print(f"  Max Drawdown:      {v3_metrics.max_drawdown_pct:.2f}%")
print(f"  Total Return:      {v3_metrics.total_return_pct:+.1f}%")
print(f"  Final Balance:     ${v3_metrics.final_balance:,.2f}")
print(f"  Trades/Day:        {v3_metrics.trades_per_day:.2f}")
print(f"  Avg Bars in Trade: {v3_metrics.avg_bars_in_trade:.1f} (4H bars)")
print(f"  Avg MFE:           {v3_metrics.avg_mfe_pips:.1f} pips")
print(f"  Avg MAE:           {v3_metrics.avg_mae_pips:.1f} pips")
print("=" * 65)

---
## Step 4 — Equity Curve & Trade Analysis

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 2, figsize=(18, 14))
fig.suptitle(
    f'WaveBot V3 Swing Reversal — {INSTRUMENT}  |  '
    f'${v3_config["initial_balance"]:,.0f} → ${v3_metrics.final_balance:,.2f}  '
    f'({v3_metrics.total_return_pct:+.1f}%)',
    fontsize=14, fontweight='bold'
)

# --- 1. Equity Curve ---
ax = axes[0, 0]
equity = v3_metrics.equity_curve
ax.plot(equity, color='#2962FF', linewidth=1.2)
ax.axhline(y=v3_config['initial_balance'], color='gray', linestyle='--', alpha=0.5)
ax.fill_between(
    range(len(equity)), v3_config['initial_balance'], equity,
    where=[e > v3_config['initial_balance'] for e in equity],
    alpha=0.15, color='green'
)
ax.fill_between(
    range(len(equity)), v3_config['initial_balance'], equity,
    where=[e < v3_config['initial_balance'] for e in equity],
    alpha=0.15, color='red'
)
ax.set_title('Equity Curve')
ax.set_ylabel('Balance ($)')
ax.grid(True, alpha=0.3)

# --- 2. Drawdown ---
ax = axes[0, 1]
eq_arr = np.array(equity)
peak = np.maximum.accumulate(eq_arr)
dd_pct = (peak - eq_arr) / peak * 100
ax.fill_between(range(len(dd_pct)), 0, dd_pct, color='red', alpha=0.4)
ax.set_title('Drawdown (%)')
ax.set_ylabel('Drawdown %')
ax.invert_yaxis()
ax.grid(True, alpha=0.3)

# --- 3. P&L per Trade ---
ax = axes[1, 0]
if v3_trades:
    pnls = [t['pnl_pips'] for t in v3_trades]
    colors = ['green' if p > 0 else 'red' for p in pnls]
    ax.bar(range(len(pnls)), pnls, color=colors, width=1.0, alpha=0.7)
    ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_title('P&L per Trade (pips)')
ax.set_ylabel('Pips')
ax.grid(True, alpha=0.3)

# --- 4. Cumulative Pips ---
ax = axes[1, 1]
if v3_trades:
    cum_pips = np.cumsum(pnls)
    ax.plot(cum_pips, color='#2962FF', linewidth=1.2)
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_title('Cumulative Pips')
ax.set_ylabel('Pips')
ax.grid(True, alpha=0.3)

# --- 5. P&L Distribution ---
ax = axes[2, 0]
if v3_trades:
    ax.hist(pnls, bins=30, color='#2962FF', alpha=0.7, edgecolor='white')
    ax.axvline(x=0, color='red', linewidth=1)
    ax.axvline(x=np.mean(pnls), color='green', linewidth=1.5, linestyle='--',
               label=f'Mean: {np.mean(pnls):.1f} pips')
    ax.set_xlabel('Pips')
    ax.legend()
ax.set_title('P&L Distribution')
ax.grid(True, alpha=0.3)

# --- 6. Exit Reasons ---
ax = axes[2, 1]
if v3_trades:
    reasons = {}
    for t in v3_trades:
        r = t.get('exit_reason', 'Unknown')
        reasons[r] = reasons.get(r, 0) + 1
    labels = list(reasons.keys())
    counts = list(reasons.values())
    colors_pie = ['#2962FF', '#FF6D00', '#00C853', '#D50000', '#AA00FF']
    ax.pie(counts, labels=labels, autopct='%1.1f%%',
           colors=colors_pie[:len(labels)], startangle=90)
    ax.set_title('Exit Reasons')

plt.tight_layout()
plt.savefig(f"{DRIVE_RESULTS_V3}/v3_charts_{INSTRUMENT}.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"\n💾 Charts saved to Drive.")

---
## Step 5 — Trade Log

In [ ]:
trade_df = pd.DataFrame(v3_trades)

display_cols = [
    'id', 'direction', 'entry_time', 'exit_time', 'entry_price',
    'exit_price', 'stop_loss', 'pnl_pips', 'pnl_dollars',
    'exit_reason', 'pattern_type', 'daily_context', 'double_down_count',
]
available_cols = [c for c in display_cols if c in trade_df.columns]

print(f"Total V3 trades: {len(trade_df)}")

if not trade_df.empty:
    print(f"\n--- First 15 trades ---")
    display(trade_df[available_cols].head(15))

    print(f"\n--- Last 15 trades ---")
    display(trade_df[available_cols].tail(15))

    # Double-down trades
    dd_trades = trade_df[trade_df['double_down_count'] > 0]
    if not dd_trades.empty:
        print(f"\n--- Double-Down Trades ({len(dd_trades)}) ---")
        display(dd_trades[available_cols])

    # Save
    trade_df.to_csv(f"{DRIVE_RESULTS_V3}/v3_trades_{INSTRUMENT}.csv", index=False)
    print(f"\n💾 Trade log saved: v3_trades_{INSTRUMENT}.csv")
else:
    print("\n⚠️  No trades generated. Try adjusting parameters (lower min_body_ratio, wider atr_multiplier).")

---
## Step 6 — V3 Go / No-Go

V3 has **different** pass criteria than V1/V2:
- Fewer trades expected (this is by design)
- Higher win rate target (60%+ expected)
- Longer hold times (4H bars)

In [ ]:
print("=" * 65)
print("  V3 SWING REVERSAL — GO / NO-GO DECISION")
print("=" * 65)

checks = {
    "Win Rate ≥ 60%":         (v3_metrics.win_rate >= 0.60, f"{v3_metrics.win_rate:.1%}"),
    "Profit Factor ≥ 1.5":    (v3_metrics.profit_factor >= 1.5, f"{v3_metrics.profit_factor:.2f}"),
    "Max Drawdown < 12%":     (v3_metrics.max_drawdown_pct < 12, f"{v3_metrics.max_drawdown_pct:.2f}%"),
    "Sharpe Ratio ≥ 0.8":     (v3_metrics.sharpe_ratio >= 0.8, f"{v3_metrics.sharpe_ratio:.2f}"),
    "Avg R:R ≥ 1.5":          (v3_metrics.avg_rr_achieved >= 1.5, f"{v3_metrics.avg_rr_achieved:.2f}"),
    "Total Trades ≥ 20":      (v3_metrics.total_trades >= 20, f"{v3_metrics.total_trades}"),
}

all_pass = True
for name, (passed, value) in checks.items():
    icon = "✅" if passed else "❌"
    status = "PASS" if passed else "FAIL"
    print(f"  {icon} {name:<28} {value:>10}   [{status}]")
    if not passed:
        all_pass = False

print("=" * 65)
if all_pass:
    print("  ✅ DECISION: GO — V3 looks good. Proceed to paper trading.")
else:
    failing = sum(1 for _, (p, _) in checks.items() if not p)
    if failing <= 2:
        print("  ⚠️  DECISION: CAUTION — Tune ATR multiplier or body ratio.")
    else:
        print("  ❌ DECISION: NO-GO — V3 needs parameter tuning.")
print("=" * 65)

---
## Step 7 — V1 vs V3 Comparison

Load V1 results from Drive (if available) for side-by-side comparison.

In [ ]:
import pickle

v1_checkpoint = f"{DRIVE_BASE}/checkpoints/backtest_default_{INSTRUMENT}.pkl"
v1_metrics = None

if os.path.exists(v1_checkpoint):
    with open(v1_checkpoint, 'rb') as f:
        v1_data = pickle.load(f)
    v1_metrics = v1_data.get('metrics')
    v1_trades = v1_data.get('trades', [])
    print("✅ Loaded V1 results from Drive.")
else:
    print("⚠️  No V1 results found on Drive. Run the V1 notebook first for comparison.")

if v1_metrics:
    print("\n" + "=" * 70)
    print(f"  {'METRIC':<28} {'V1 (Wave Engine)':<20} {'V3 (Swing Reversal)':<20}")
    print("=" * 70)

    comparisons = [
        ("Total Trades",       f"{v1_metrics.total_trades}",           f"{v3_metrics.total_trades}"),
        ("Win Rate",           f"{v1_metrics.win_rate:.1%}",           f"{v3_metrics.win_rate:.1%}"),
        ("Profit Factor",      f"{v1_metrics.profit_factor:.2f}",      f"{v3_metrics.profit_factor:.2f}"),
        ("Total Pips",         f"{v1_metrics.total_pips:+.1f}",        f"{v3_metrics.total_pips:+.1f}"),
        ("Avg Win (pips)",     f"{v1_metrics.avg_win_pips:.1f}",       f"{v3_metrics.avg_win_pips:.1f}"),
        ("Avg Loss (pips)",    f"{v1_metrics.avg_loss_pips:.1f}",      f"{v3_metrics.avg_loss_pips:.1f}"),
        ("Avg R:R Achieved",   f"{v1_metrics.avg_rr_achieved:.2f}",    f"{v3_metrics.avg_rr_achieved:.2f}"),
        ("Sharpe Ratio",       f"{v1_metrics.sharpe_ratio:.2f}",       f"{v3_metrics.sharpe_ratio:.2f}"),
        ("Max Drawdown",       f"{v1_metrics.max_drawdown_pct:.2f}%",  f"{v3_metrics.max_drawdown_pct:.2f}%"),
        ("Total Return",       f"{v1_metrics.total_return_pct:+.1f}%", f"{v3_metrics.total_return_pct:+.1f}%"),
        ("Final Balance",      f"${v1_metrics.final_balance:,.2f}",    f"${v3_metrics.final_balance:,.2f}"),
        ("Trades/Day",         f"{v1_metrics.trades_per_day:.1f}",     f"{v3_metrics.trades_per_day:.2f}"),
        ("Avg Bars in Trade",  f"{v1_metrics.avg_bars_in_trade:.1f}",  f"{v3_metrics.avg_bars_in_trade:.1f}"),
    ]

    for name, v1_val, v3_val in comparisons:
        print(f"  {name:<28} {v1_val:<20} {v3_val:<20}")

    print("=" * 70)
    print("\n  V1: High-frequency, multi-timeframe wave alignment (M5-driven)")
    print("  V3: Low-frequency, 4H reversal pattern with trailing stop")
    print("      → Fewer trades, longer holds, targets big moves")

In [ ]:
RUN_OPTIMIZATION = False  # Set to True to run

if RUN_OPTIMIZATION:
    from itertools import product

    param_grid = {
        "atr_multiplier": [1.5, 2.0, 2.5, 3.0],
        "min_body_ratio": [0.2, 0.3, 0.4],
        "sl_buffer_pips": [1.0, 2.0, 3.0],
    }

    keys = list(param_grid.keys())
    combos = list(product(*param_grid.values()))
    print(f"Testing {len(combos)} combinations...\n")

    results = []
    for i, vals in enumerate(combos):
        params = dict(zip(keys, vals))
        cfg = {**v3_config, **params}

        try:
            eng = V3BacktestEngine(**cfg)
            trades = eng.run(h4_df.copy(), d_df.copy())
            m = calculate_metrics(
                trades, cfg['initial_balance'],
                INSTRUMENT, DATA_START[:10], DATA_END[:10], trading_days
            )

            score = (
                m.profit_factor * 0.3 +
                m.win_rate * 0.25 +
                m.sharpe_ratio * 0.25 +
                (1 - m.max_drawdown_pct / 100) * 0.2
            ) if m.total_trades >= 10 else -1

            results.append({**params, 'score': score, 'trades': m.total_trades,
                           'wr': m.win_rate, 'pf': m.profit_factor,
                           'sharpe': m.sharpe_ratio, 'dd': m.max_drawdown_pct,
                           'balance': m.final_balance})

            print(f"  [{i+1:>3}/{len(combos)}] ATR={params['atr_multiplier']:.1f} "
                  f"Body={params['min_body_ratio']:.1f} Buf={params['sl_buffer_pips']:.1f} | "
                  f"Score={score:.3f} WR={m.win_rate:.0%} PF={m.profit_factor:.2f} "
                  f"Trades={m.total_trades} ${m.final_balance:,.2f}")
        except Exception as e:
            print(f"  [{i+1:>3}/{len(combos)}] ERROR: {e}")
            results.append({**params, 'score': -999})

    # Show top 5
    ranked = sorted(results, key=lambda x: x['score'], reverse=True)
    print("\n" + "=" * 70)
    print("  TOP 5 V3 PARAMETER SETS")
    print("=" * 70)
    for i, r in enumerate(ranked[:5]):
        print(f"  #{i+1}: ATR={r['atr_multiplier']:.1f} Body={r['min_body_ratio']:.1f} "
              f"Buf={r['sl_buffer_pips']:.1f} → Score={r['score']:.3f} "
              f"WR={r.get('wr',0):.0%} PF={r.get('pf',0):.2f} ${r.get('balance',0):,.2f}")

    # Save
    opt_df = pd.DataFrame(ranked)
    opt_df.to_csv(f"{DRIVE_RESULTS_V3}/v3_optimization_{INSTRUMENT}.csv", index=False)
    print(f"\n💾 Optimization results saved.")
else:
    print("Optimization skipped. Set RUN_OPTIMIZATION = True to run.")

---
## Step 9 — Save Final Results

In [ ]:
import json

# Save summary
summary = {
    "strategy": "swing_reversal_v3",
    "instrument": INSTRUMENT,
    "data_period": f"{DATA_START[:10]} to {DATA_END[:10]}",
    "config": {k: v for k, v in v3_config.items() if not callable(v)},
    "metrics": {
        "total_trades": v3_metrics.total_trades,
        "win_rate": round(v3_metrics.win_rate, 4),
        "profit_factor": round(v3_metrics.profit_factor, 2),
        "sharpe_ratio": round(v3_metrics.sharpe_ratio, 2),
        "calmar_ratio": round(v3_metrics.calmar_ratio, 2),
        "max_drawdown_pct": round(v3_metrics.max_drawdown_pct, 2),
        "total_return_pct": round(v3_metrics.total_return_pct, 2),
        "final_balance": round(v3_metrics.final_balance, 2),
        "avg_rr_achieved": round(v3_metrics.avg_rr_achieved, 2),
        "trades_per_day": round(v3_metrics.trades_per_day, 4),
        "avg_bars_in_trade": round(v3_metrics.avg_bars_in_trade, 1),
    },
}

with open(f"{DRIVE_RESULTS_V3}/v3_summary_{INSTRUMENT}.json", 'w') as f:
    json.dump(summary, f, indent=2)

# Save full results with pickle
with open(f"{DRIVE_RESULTS_V3}/v3_backtest_{INSTRUMENT}.pkl", 'wb') as f:
    pickle.dump({
        'trades': v3_trades,
        'metrics': v3_metrics,
        'config': v3_config,
    }, f)

print("💾 Saved to Drive:")
print(f"   {DRIVE_RESULTS_V3}/v3_summary_{INSTRUMENT}.json")
print(f"   {DRIVE_RESULTS_V3}/v3_backtest_{INSTRUMENT}.pkl")
print(f"   {DRIVE_RESULTS_V3}/v3_trades_{INSTRUMENT}.csv")
print(f"   {DRIVE_RESULTS_V3}/v3_charts_{INSTRUMENT}.png")
print("\n✅ All V3 results saved.")